# Map VASA data onto multiome 

In [ ]:
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import json
import cellmapper
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import marsilea as ma
import marsilea.plotter as mp
from pathlib import Path

# Global Scanpy settings
sc.settings.verbosity = 2               # Show progress
sc.settings.set_figure_params(
    dpi=150,                            # High-res output
    dpi_save=300,                       # High-res when saving
    format='pdf',                       # or 'pdf', 'svg'
    facecolor='white',                  # or 'none' for transparent
    frameon=False,                      # No outer frame
    vector_friendly=True,               # No rasterization warnings
    fontsize=10,                        # Base font size
    figsize=(4, 4),                     # Default figure size
    transparent=True                    # Transparent background if saving
)

mpl.rcParams.update({
    "svg.fonttype": "none",       
    "pdf.fonttype": 42,           
    "legend.fontsize": 6,
    "axes.titlesize": 6,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
})

# Write to output directory
output_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/grn")
output_dir.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = output_dir / "figures"

## Load data 

In [ ]:
adata = sc.read(output_dir / "tf_ko_panel_control_vasa_combined_pca_v3.h5ad")

In [ ]:
adata_query = adata[adata.obs["dataset"].isin(["Multiome"])]
adata_ref = adata[adata.obs["dataset"].isin(["VASA"])]

In [ ]:
cmap = cellmapper.CellMapper(adata_query, adata_ref).map(
    obs_keys=["annotation", "milestones"], use_rep="X_pca", knn_method="pynndescent"
)
cmap

In [ ]:
sc.pl.embedding(adata_query, basis="X_umap", color=["annotation_pred", "annotation_conf", "milestones_pred", "milestones_conf"], legend_loc="on data")

In [ ]:
adata_multiome = sc.read(output_dir / "EEC_multiome_rna_matched_subset.h5ad")

In [ ]:
# Normalize and log transform
sc.pp.normalize_total(adata_multiome, target_sum=1e4)
sc.pp.log1p(adata_multiome)
adata_multiome.layers["lognorm"] = adata_multiome.X.copy()

In [ ]:
adata_query.obs_names = adata_query.obs_names.str.replace("-Multiome", "")

adata_multiome.obs["annotation_pred"] = adata_query.obs["annotation_pred"]
adata_multiome.obs["annotation_conf"] = adata_query.obs["annotation_conf"]
adata_multiome.obs["milestones_pred"] = adata_query.obs["milestones_pred"]
adata_multiome.obs["milestones_conf"] = adata_query.obs["milestones_conf"]

In [ ]:
sc.pl.embedding(adata_multiome, basis="X_umap_harmony", color=["annotation", "annotation_pred", "annotation_conf", "milestones_pred", "milestones_conf", "GHRL", "NEUROG3", "GIP", "SST", "MLN", "PERCC1"])

In [ ]:
adata_multiome.write(output_dir / "multiome_mapped_vasa_milestones_v3.h5ad")

## Plot label transfer results on multiome UMAP

In [ ]:
# Load data
adata_multiome = sc.read(output_dir / "multiome_mapped_vasa_milestones_v3.h5ad")

In [ ]:
# Load mapping
mapping = dict(zip(["100", "140", "186", "120", "59", "170", "165", "17"], 
                   ["X cells","D cells","X/D/I/K cells","EC/X/D/I/K cells",
                    "X/D/I/K cells","NGN3+ cells", "EC cells", "I/K cells"]))
# Transfer annotation
adata_multiome.obs["milestones_anno_pred"] = adata_multiome.obs["milestones_pred"].map(mapping)

In [ ]:
annotation_palette = {
    "eecs_milestones_pred": {
        # Pure populations
        "NGN3+ cells": "#9467bd",   # purple (progenitors)
        "EC/X/D/I/K cells": "#fdae61",    # warm orange blend
        "EC cells": "#d62728",      # strong red
        "X/D/I/K cells": "#c2a5cf",       # lavender blend
        "X cells": "#e377c2",       # magenta
        "D cells": "#17becf",       # cyan
        "I/K cells": "#bcbd22",     # olive
    },
    "eecs_annotation_pred": {
        # EEC progenitors
        "EEC Progenitors": "#9467bd",   # purple

        # EEC subtypes 
        "EC Cells": "#d62728",            # strong red
        "X Cells": "#e377c2",             # magenta
        "D Cells": "#17becf",             # cyan / turquoise
        "I/N Cells": "#7f7f7f",           # dark grey
        "K Cells": "#bcbd22",             # olive
    }
}

In [ ]:
sc.pl.embedding(adata_multiome, basis="X_umap_harmony", color="annotation", palette=annotation_palette["eecs_annotation_pred"], legend_loc="on data", size=20, save="_multiome_mapped_vasa_annotation.pdf")

In [ ]:
sc.pl.embedding(adata_multiome, basis="X_umap_harmony", color="milestones_anno_pred", palette=annotation_palette["eecs_milestones_pred"], legend_loc="on data", size=20, save="_multiome_mapped_vasa_milestones.pdf")

In [ ]:
sc.pl.embedding(adata_multiome, basis="X_umap_harmony", color="milestones_conf", cmap="inferno", legend_loc="on data", size=20, save="_multiome_mapped_vasa_milestones_confidence.pdf")

## Plot heatmap of mean expression per milestone for perturbed TF

In [ ]:
vasa_df_norm = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.perturbed_tfs.expression.heatmap.csv", index_col=0)

perturbed_tfs = vasa_df_norm.columns.tolist()
perturbed_tfs = [tf for tf in perturbed_tfs if tf in adata_multiome.var_names]

In [ ]:
adata_multiome = sc.read(output_dir / "multiome_mapped_vasa_milestones_v3.h5ad")
adata = adata_multiome[:, perturbed_tfs].copy()

In [ ]:
# Extract expression as dense DataFrame + add obs grouping column
df = sc.get.obs_df(adata, list(adata.var_names))
df["milestones"] = adata.obs["milestones_pred"].values
df["milestones"] = df["milestones"].map(mapping)

# Aggregate mean expression per seg
df_mean = df.groupby("milestones").mean()

milestone_order = vasa_df_norm.index.tolist()
df_mean = df_mean.reindex(milestone_order)

df_norm = df_mean[vasa_df_norm.columns.tolist()].T.copy()
df_norm = df_norm.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1).T
df_norm = df_norm[vasa_df_norm.columns.tolist()]

In [ ]:
import marsilea as ma
import marsilea.plotter as mp

h = ma.Heatmap(data=df_norm, width=12, height=6, cmap="RdPu")
h.add_right(mp.Labels(df_norm.index))
h.add_bottom(mp.Labels(df_norm.columns, rotation=90))
h.render()

# Show same for TF motif accessibility

In [ ]:
adata_multiome_chromvar = sc.read(output_dir / "EEC_multiome_chromvar_matched.h5ad")
adata_multiome_chromvar = adata_multiome_chromvar[adata_multiome.obs_names,:].copy()

# Add predicted milestones
adata_multiome_chromvar.obs["annotation_pred"] = adata_multiome.obs["annotation_pred"]
adata_multiome_chromvar.obs["annotation_conf"] = adata_multiome.obs["annotation_conf"]
adata_multiome_chromvar.obs["milestones_pred"] = adata_multiome.obs["milestones_pred"]
adata_multiome_chromvar.obs["milestones_conf"] = adata_multiome.obs["milestones_conf"]

# Add UMAP from RNA 
adata_multiome_chromvar.obsm["X_umap"] = adata_multiome.obsm["X_umap_harmony"]

adata_multiome_chromvar.write(output_dir / "multiome_chromvar_mapped_vasa_milestones_v3.h5ad")

## Plot heatmap of mean motif accessibility per milestone for perturbed TF

In [ ]:
adata_multiome_chromvar = sc.read(output_dir / "multiome_chromvar_mapped_vasa_milestones_v3.h5ad")
adata_multiome_chromvar.var.index = adata_multiome_chromvar.var.index.str.split("_").str[0]

adata_multiome_chromvar.obs["milestones"] = adata_multiome_chromvar.obs["milestones_pred"].map(mapping)

In [ ]:
adata_multiome_chromvar.obs["milestones"].to_csv(output_dir / "multiome_chromvar_mapped_vasa_milestones_annotations_v3.csv")

In [ ]:
vasa_df_norm = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.scfates.perturbed_tfs.expression.heatmap.csv", index_col=0)

perturbed_tfs = vasa_df_norm.columns.tolist()
perturbed_tfs = [tf for tf in perturbed_tfs if tf in adata_multiome.var_names]

In [ ]:
# Keep TFs with motif
keep_tfs = [tf for tf in perturbed_tfs if tf in adata_multiome_chromvar.var_names]

In [ ]:
adata = adata_multiome_chromvar[:, keep_tfs].copy()

In [ ]:
# Extract expression as dense DataFrame + add obs grouping column
df = sc.get.obs_df(adata, list(adata.var_names))
df["milestones"] = adata.obs["milestones_pred"].values
df["milestones"] = df["milestones"].map(mapping)

# Aggregate mean expression per seg
df_mean = df.groupby("milestones").mean()

In [ ]:
# Extract expression as dense DataFrame + add obs grouping column
df = sc.get.obs_df(adata, list(adata.var_names))
df["milestones"] = adata.obs["milestones_pred"].values
df["milestones"] = df["milestones"].map(mapping)

# Aggregate mean expression per seg
df_mean = df.groupby("milestones").mean()

milestone_order = vasa_df_norm.index.tolist()
df_mean = df_mean.reindex(milestone_order)

df_norm = df_mean.T.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1).T
for col in perturbed_tfs:
    if col not in df_norm.columns:
        df_norm[col] = np.nan
df_norm = df_norm[vasa_df_norm.columns.tolist()].copy()

In [ ]:
df_norm.to_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/multiome.processed.eecs.perturbed_tfs.imputed_motif_acc.heatmap_v3.csv")

In [ ]:
import marsilea as ma
import marsilea.plotter as mp

h = ma.Heatmap(data=df_norm, width=12, height=6, cmap="RdPu")
h.add_right(mp.Labels(df_norm.index))
h.add_bottom(mp.Labels(df_norm.columns, rotation=90))
h.render()